In [1]:
# ============================================================
# W11 — CELL 1
# SETUP, DIRECTORIES & MODEL VERSION
# ============================================================

import os
import json
import pandas as pd
import numpy as np

from datetime import datetime

# ------------------------------------------------------------
# Create W11 directories
# ------------------------------------------------------------

os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)
os.makedirs("../artifacts", exist_ok=True)

# ------------------------------------------------------------
# Model version
# ------------------------------------------------------------

MODEL_VERSION = "aeropure_xgb_v1"

# ------------------------------------------------------------
# Champion model information
# ------------------------------------------------------------

MODEL_NAME = "AeroPure XGBoost Champion"

MODEL_TYPE = "XGBoost"

# Original W8 feature set
W8_FEATURES = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene",
    "AQI_Current",
    "AQI_Lag1",
    "AQI_3Day_Mean",
    "DayOfWeek",
    "Month"
]

# ------------------------------------------------------------
# Display setup
# ------------------------------------------------------------

print("AeroPure — Week 11")
print("=" * 50)

print("Model version :", MODEL_VERSION)
print("Model name    :", MODEL_NAME)
print("Model type    :", MODEL_TYPE)

print("\nFeature count :", len(W8_FEATURES))

print("\nFeatures:")
for i, feature in enumerate(W8_FEATURES, start=1):
    print(f"{i:2d}. {feature}")

AeroPure — Week 11
Model version : aeropure_xgb_v1
Model name    : AeroPure XGBoost Champion
Model type    : XGBoost

Feature count : 17

Features:
 1. PM2.5
 2. PM10
 3. NO
 4. NO2
 5. NOx
 6. NH3
 7. CO
 8. SO2
 9. O3
10. Benzene
11. Toluene
12. Xylene
13. AQI_Current
14. AQI_Lag1
15. AQI_3Day_Mean
16. DayOfWeek
17. Month


In [2]:
# ============================================================
# W11 — CELL 2
# LOAD DATA & BUILD CHAMPION FEATURE MATRICES
# ============================================================

# Load W9 datasets
train_w9 = pd.read_csv("../data/train_fe_w9.csv")
test_w9 = pd.read_csv("../data/test_fe_w9.csv")

# Convert Date
train_w9["Date"] = pd.to_datetime(train_w9["Date"])
test_w9["Date"] = pd.to_datetime(test_w9["Date"])

# ------------------------------------------------------------
# Recreate classification target
# ------------------------------------------------------------

TARGET = "Hazardous_Tomorrow"

train_w9[TARGET] = (
    train_w9["Tomorrow_AQI"] >= 301
).astype(int)

test_w9[TARGET] = (
    test_w9["Tomorrow_AQI"] >= 301
).astype(int)

# ------------------------------------------------------------
# Build champion feature matrices
# ------------------------------------------------------------

X_train_w11 = train_w9[W8_FEATURES].copy()
X_test_w11 = test_w9[W8_FEATURES].copy()

y_train_w11 = train_w9[TARGET].copy()
y_test_w11 = test_w9[TARGET].copy()

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("W11 DATASET VALIDATION")
print("=" * 50)

print("\nTraining shape:")
print("X_train:", X_train_w11.shape)
print("y_train:", y_train_w11.shape)

print("\nTesting shape:")
print("X_test :", X_test_w11.shape)
print("y_test :", y_test_w11.shape)

print("\nFeature count:", len(W8_FEATURES))

print("\nMissing values in training:")
print(X_train_w11.isnull().sum().sum())

print("\nMissing values in testing:")
print(X_test_w11.isnull().sum().sum())

print("\nTarget distribution — Training:")
print(y_train_w11.value_counts())

print("\nTarget distribution — Testing:")
print(y_test_w11.value_counts())

print("\nFeature schema verified:")
print(
    list(X_train_w11.columns) == W8_FEATURES
)

W11 DATASET VALIDATION

Training shape:
X_train: (15636, 17)
y_train: (15636,)

Testing shape:
X_test : (8681, 17)
y_test : (8681,)

Feature count: 17

Missing values in training:
30279

Missing values in testing:
12006

Target distribution — Training:
Hazardous_Tomorrow
0    12642
1     2994
Name: count, dtype: int64

Target distribution — Testing:
Hazardous_Tomorrow
0    8082
1     599
Name: count, dtype: int64

Feature schema verified:
True


In [3]:
# ============================================================
# W11 — CELL 3
# TRAIN FINAL CHAMPION MODEL
# ============================================================

from xgboost import XGBClassifier
import joblib

# ------------------------------------------------------------
# Final W8 XGBoost parameters
# ------------------------------------------------------------

xgb_params_w11 = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": 4.2224,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}

# ------------------------------------------------------------
# Create champion model
# ------------------------------------------------------------

champion_model = XGBClassifier(
    **xgb_params_w11
)

# ------------------------------------------------------------
# Train on ALL available training data
# ------------------------------------------------------------

print("Training AeroPure Champion Model...")
print("=" * 55)

champion_model.fit(
    X_train_w11,
    y_train_w11
)

print("Training completed successfully.")

# ------------------------------------------------------------
# Test prediction
# ------------------------------------------------------------

champion_test_pred = champion_model.predict(
    X_test_w11
)

champion_test_prob = champion_model.predict_proba(
    X_test_w11
)[:, 1]

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

model_path = (
    f"../models/{MODEL_VERSION}.joblib"
)

joblib.dump(
    champion_model,
    model_path
)

print("\nModel saved successfully:")
print(model_path)

# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

print("\nModel verification")
print("=" * 30)

print("Model type:",
      type(champion_model).__name__)

print("Number of features:",
      len(champion_model.feature_names_in_))

print("Prediction count:",
      len(champion_test_pred))

print("Probability range:",
      round(champion_test_prob.min(), 4),
      "to",
      round(champion_test_prob.max(), 4))

Training AeroPure Champion Model...
Training completed successfully.

Model saved successfully:
../models/aeropure_xgb_v1.joblib

Model verification
Model type: XGBClassifier
Number of features: 17
Prediction count: 8681
Probability range: 1e-04 to 0.999


In [4]:
# ============================================================
# W11 — CELL 4
# SAVE MODEL METADATA & FEATURE SCHEMA
# ============================================================

import json
from datetime import datetime

# ------------------------------------------------------------
# Model metadata
# ------------------------------------------------------------

model_metadata = {
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "model_type": MODEL_TYPE,

    "target": "Hazardous_Tomorrow",
    "hazardous_threshold": 301,

    "feature_count": len(W8_FEATURES),
    "features": W8_FEATURES,

    "training_rows": len(X_train_w11),
    "test_rows": len(X_test_w11),

    "training_start": str(
        train_w9["Date"].min().date()
    ),

    "training_end": str(
        train_w9["Date"].max().date()
    ),

    "test_start": str(
        test_w9["Date"].min().date()
    ),

    "test_end": str(
        test_w9["Date"].max().date()
    ),

    "hyperparameters": xgb_params_w11,

    "created_at": datetime.now().isoformat(),

    "artifact": model_path
}

# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------

metadata_path = (
    f"../models/{MODEL_VERSION}_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        model_metadata,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Save feature schema separately
# ------------------------------------------------------------

schema = {
    "model_version": MODEL_VERSION,
    "feature_count": len(W8_FEATURES),
    "features": W8_FEATURES,
    "target": "Hazardous_Tomorrow"
}

schema_path = (
    f"../models/{MODEL_VERSION}_feature_schema.json"
)

with open(
    schema_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        schema,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("MODEL METADATA & SCHEMA SAVED")
print("=" * 55)

print("\nMetadata:")
print(metadata_path)

print("\nFeature schema:")
print(schema_path)

print("\nArtifacts created:")
print("✓", model_path)
print("✓", metadata_path)
print("✓", schema_path)

print("\nFrozen feature count:", len(W8_FEATURES))

MODEL METADATA & SCHEMA SAVED

Metadata:
../models/aeropure_xgb_v1_metadata.json

Feature schema:
../models/aeropure_xgb_v1_feature_schema.json

Artifacts created:
✓ ../models/aeropure_xgb_v1.joblib
✓ ../models/aeropure_xgb_v1_metadata.json
✓ ../models/aeropure_xgb_v1_feature_schema.json

Frozen feature count: 17


In [5]:
# ============================================================
# W11 — CELL 5
# RELOAD & VERIFY SAVED MODEL
# ============================================================

import joblib
import numpy as np

# ------------------------------------------------------------
# Load model from disk
# ------------------------------------------------------------

loaded_model = joblib.load(
    model_path
)

print("Saved model loaded successfully.")
print("Model type:", type(loaded_model).__name__)

# ------------------------------------------------------------
# Generate predictions using loaded model
# ------------------------------------------------------------

loaded_pred = loaded_model.predict(
    X_test_w11
)

loaded_prob = loaded_model.predict_proba(
    X_test_w11
)[:, 1]

# ------------------------------------------------------------
# Compare with original in-memory model
# ------------------------------------------------------------

predictions_identical = np.array_equal(
    champion_test_pred,
    loaded_pred
)

probabilities_identical = np.allclose(
    champion_test_prob,
    loaded_prob,
    rtol=1e-10,
    atol=1e-12
)

# ------------------------------------------------------------
# Verification output
# ------------------------------------------------------------

print("\nMODEL REPRODUCIBILITY CHECK")
print("=" * 50)

print(
    "Predictions identical    :",
    predictions_identical
)

print(
    "Probabilities identical  :",
    probabilities_identical
)

print(
    "Prediction count         :",
    len(loaded_pred)
)

print(
    "Probability range        :",
    round(loaded_prob.min(), 4),
    "to",
    round(loaded_prob.max(), 4)
)

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

if predictions_identical and probabilities_identical:
    print("\n✅ MODEL ARTIFACT VERIFIED SUCCESSFULLY")
else:
    print("\n❌ MODEL VERIFICATION FAILED")

Saved model loaded successfully.
Model type: XGBClassifier

MODEL REPRODUCIBILITY CHECK
Predictions identical    : True
Probabilities identical  : True
Prediction count         : 8681
Probability range        : 1e-04 to 0.999

✅ MODEL ARTIFACT VERIFIED SUCCESSFULLY


In [6]:
# Cell 6 — Reusable AeroPure Prediction Function

import pandas as pd
import numpy as np
import joblib

def predict_hazardous_day(input_df, model, feature_schema):
    """
    Predict whether tomorrow will be hazardous.

    Parameters
    ----------
    input_df : pandas DataFrame
        Input data containing the required AeroPure features.
        
    model : trained XGBoost model
        Saved AeroPure champion model.
        
    feature_schema : list
        Required feature names in the correct order.

    Returns
    -------
    pandas DataFrame
        Prediction probability and hazardous-day prediction.
    """

    # Check for missing features
    missing_features = [
        feature for feature in feature_schema
        if feature not in input_df.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing required features: {missing_features}"
        )

    # Keep only required features and correct order
    X_input = input_df[feature_schema].copy()

    # Generate probability
    probability = model.predict_proba(X_input)[:, 1]

    # Convert probability into class
    prediction = (probability >= 0.5).astype(int)

    # Create result
    result = pd.DataFrame({
        "Hazard_Probability": probability,
        "Hazardous_Tomorrow": prediction
    })

    # Add readable status
    result["Prediction"] = np.where(
        result["Hazardous_Tomorrow"] == 1,
        "Hazardous",
        "Not Hazardous"
    )

    return result


print("Reusable prediction function created successfully.")

Reusable prediction function created successfully.


In [7]:
# Cell 7 — Test Reusable Prediction Function

# Load the feature schema
with open("../models/aeropure_xgb_v1_feature_schema.json", "r") as f:
    feature_schema_data = json.load(f)

# Extract feature list
feature_schema = feature_schema_data["features"]

# Run prediction on a sample of test data
sample_input = X_test_w11.head(10)

sample_predictions = predict_hazardous_day(
    sample_input,
    loaded_model,
    feature_schema
)

# Display input + predictions
display(sample_predictions)

print("\nPrediction function test completed successfully.")
print("Number of predictions:", len(sample_predictions))

,Hazard_Probability,Hazardous_Tomorrow,Prediction
0,0.979986,1,Hazardous
1,0.000545,0,Not Hazardous
2,0.000416,0,Not Hazardous
3,0.000183,0,Not Hazardous
4,0.000763,0,Not Hazardous
5,0.003552,0,Not Hazardous
6,0.005986,0,Not Hazardous
7,0.001464,0,Not Hazardous
8,0.865459,1,Hazardous
9,0.002535,0,Not Hazardous



Prediction function test completed successfully.
Number of predictions: 10


In [8]:
# Cell 8 — Create Training Feature Drift Baseline

import json
import pandas as pd
import numpy as np

# Use the 17 champion-model features
baseline_features = X_train_w11[W8_FEATURES].copy()

# Create baseline statistics
baseline_stats = pd.DataFrame({
    "Feature": W8_FEATURES,
    "Mean": [baseline_features[f].mean() for f in W8_FEATURES],
    "Std": [baseline_features[f].std() for f in W8_FEATURES],
    "Min": [baseline_features[f].min() for f in W8_FEATURES],
    "Median": [baseline_features[f].median() for f in W8_FEATURES],
    "Max": [baseline_features[f].max() for f in W8_FEATURES]
})

# Save baseline
baseline_path = "../results/aeropure_xgb_v1_drift_baseline.csv"
baseline_stats.to_csv(baseline_path, index=False)

print("Feature drift baseline created successfully.")
print("Number of features:", len(baseline_stats))
print("Saved to:", baseline_path)

display(baseline_stats)

Feature drift baseline created successfully.
Number of features: 17
Saved to: ../results/aeropure_xgb_v1_drift_baseline.csv


,Feature,Mean,Std,Min,Median,Max
0,PM2.5,78.374191,68.779206,1.72,57.660000,949.990000
1,PM10,140.671680,98.313737,0.02,111.850000,917.080000
2,NO,18.663427,23.509387,0.06,10.430000,287.140000
3,NO2,32.094207,25.109493,0.01,25.600000,266.460000
4,NOx,33.839435,32.463797,0.00,24.940000,293.100000
5,NH3,27.834947,30.469126,0.02,19.885000,352.890000
6,CO,2.732769,7.893568,0.00,1.010000,175.810000
7,SO2,14.708702,17.830506,0.26,8.890000,171.320000
8,O3,36.412293,22.898876,0.10,32.260000,257.730000
9,Benzene,2.669005,6.905392,0.00,1.220000,391.880000


In [9]:
# Cell 9 — Population Stability Index (PSI)

def calculate_psi(expected, actual, bins=10):
    """
    Calculate Population Stability Index (PSI).

    expected = training/baseline feature values
    actual   = new/current feature values
    """

    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    # Create bins using the expected/training distribution
    breakpoints = np.unique(
        np.percentile(
            expected,
            np.linspace(0, 100, bins + 1)
        )
    )

    # If there are not enough unique values, use a simple range
    if len(breakpoints) < 3:
        min_value = min(expected.min(), actual.min())
        max_value = max(expected.max(), actual.max())

        if min_value == max_value:
            return 0.0

        breakpoints = np.linspace(
            min_value,
            max_value,
            bins + 1
        )

    # Extend boundaries so all actual values are included
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    expected_bins = pd.cut(
        expected,
        bins=breakpoints,
        include_lowest=True
    )

    actual_bins = pd.cut(
        actual,
        bins=breakpoints,
        include_lowest=True
    )

    expected_dist = expected_bins.value_counts(
        normalize=True,
        sort=False
    )

    actual_dist = actual_bins.value_counts(
        normalize=True,
        sort=False
    )

    # Avoid division by zero
    epsilon = 1e-6

    expected_dist = expected_dist.clip(lower=epsilon)
    actual_dist = actual_dist.clip(lower=epsilon)

    # PSI formula
    psi = np.sum(
        (actual_dist - expected_dist)
        * np.log(actual_dist / expected_dist)
    )

    return float(psi)


def interpret_psi(psi):
    """
    Interpret PSI score.
    """

    if psi < 0.10:
        return "No significant drift"
    elif psi < 0.25:
        return "Moderate drift"
    else:
        return "Significant drift"


print("PSI drift detection functions created successfully.")

PSI drift detection functions created successfully.


In [10]:
# Cell 10 — Calculate PSI for All Features

psi_results = []

for feature in W8_FEATURES:

    expected = X_train_w11[feature]
    actual = X_test_w11[feature]

    psi_value = calculate_psi(
        expected,
        actual,
        bins=10
    )

    psi_results.append({
        "Feature": feature,
        "PSI": psi_value,
        "Interpretation": interpret_psi(psi_value)
    })

# Create results table
psi_results_df = pd.DataFrame(psi_results)

# Sort from highest to lowest drift
psi_results_df = psi_results_df.sort_values(
    "PSI",
    ascending=False
).reset_index(drop=True)

# Save results
psi_path = "../results/aeropure_xgb_v1_psi_test.csv"

psi_results_df.to_csv(
    psi_path,
    index=False
)

print("PSI analysis completed successfully.")
print("Features analyzed:", len(psi_results_df))
print("Saved to:", psi_path)

display(psi_results_df)

PSI analysis completed successfully.
Features analyzed: 17
Saved to: ../results/aeropure_xgb_v1_psi_test.csv


,Feature,PSI,Interpretation
0,PM10,0.422230,Significant drift
1,PM2.5,0.407831,Significant drift
2,AQI_3Day_Mean,0.404712,Significant drift
3,AQI_Current,0.371813,Significant drift
4,AQI_Lag1,0.368323,Significant drift
5,CO,0.286147,Significant drift
6,Xylene,0.237311,Moderate drift
7,NO2,0.185249,Moderate drift
8,NH3,0.174602,Moderate drift
9,SO2,0.098163,No significant drift


In [11]:
# Cell 11 — AeroPure Drift Alert Report

# Count drift categories
significant_count = (
    psi_results_df["Interpretation"] == "Significant drift"
).sum()

moderate_count = (
    psi_results_df["Interpretation"] == "Moderate drift"
).sum()

no_drift_count = (
    psi_results_df["Interpretation"] == "No significant drift"
).sum()

# Overall status
if significant_count > 0:
    overall_status = "SIGNIFICANT DRIFT DETECTED"
elif moderate_count > 0:
    overall_status = "MODERATE DRIFT DETECTED"
else:
    overall_status = "NO SIGNIFICANT DRIFT"

# Create summary
drift_summary = pd.DataFrame({
    "Metric": [
        "Total Features",
        "Significant Drift Features",
        "Moderate Drift Features",
        "No Significant Drift Features",
        "Overall Drift Status"
    ],
    "Value": [
        len(psi_results_df),
        significant_count,
        moderate_count,
        no_drift_count,
        overall_status
    ]
})

# Display summary
print("AeroPure Feature Drift Monitoring Report")
print("=" * 50)

display(drift_summary)

print("\nFeatures with Significant Drift:")
display(
    psi_results_df[
        psi_results_df["Interpretation"] == "Significant drift"
    ]
)

print("\nFeatures with Moderate Drift:")
display(
    psi_results_df[
        psi_results_df["Interpretation"] == "Moderate drift"
    ]
)

# Save report
drift_report_path = "../results/aeropure_xgb_v1_drift_report.csv"

drift_summary.to_csv(
    drift_report_path,
    index=False
)

print("\nDrift report saved to:", drift_report_path)

AeroPure Feature Drift Monitoring Report


,Metric,Value
0,Total Features,17
1,Significant Drift Features,6
2,Moderate Drift Features,3
3,No Significant Drift Features,8
4,Overall Drift Status,SIGNIFICANT DRIFT DETECTED



Features with Significant Drift:


,Feature,PSI,Interpretation
0,PM10,0.422230,Significant drift
1,PM2.5,0.407831,Significant drift
2,AQI_3Day_Mean,0.404712,Significant drift
3,AQI_Current,0.371813,Significant drift
4,AQI_Lag1,0.368323,Significant drift
5,CO,0.286147,Significant drift



Features with Moderate Drift:


,Feature,PSI,Interpretation
6,Xylene,0.237311,Moderate drift
7,NO2,0.185249,Moderate drift
8,NH3,0.174602,Moderate drift



Drift report saved to: ../results/aeropure_xgb_v1_drift_report.csv


In [12]:
# Cell 12 — Save AeroPure Drift Monitoring Configuration

drift_config = {
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "monitoring_method": "Population Stability Index (PSI)",
    
    "thresholds": {
        "no_significant_drift": 0.10,
        "moderate_drift": 0.25,
        "significant_drift": 0.25
    },
    
    "interpretation": {
        "below_0.10": "No significant drift",
        "0.10_to_below_0.25": "Moderate drift",
        "0.25_or_above": "Significant drift"
    },
    
    "features_monitored": W8_FEATURES,
    
    "baseline_file": "aeropure_xgb_v1_drift_baseline.csv",
    "psi_test_file": "aeropure_xgb_v1_psi_test.csv",
    "drift_report_file": "aeropure_xgb_v1_drift_report.csv"
}

drift_config_path = "../models/aeropure_xgb_v1_drift_config.json"

with open(drift_config_path, "w") as f:
    json.dump(drift_config, f, indent=4)

print("Drift monitoring configuration saved successfully.")
print("Configuration file:", drift_config_path)
print("Features monitored:", len(W8_FEATURES))

Drift monitoring configuration saved successfully.
Configuration file: ../models/aeropure_xgb_v1_drift_config.json
Features monitored: 17


In [13]:
# Cell 13 — Final AeroPure Model Package Verification

import os
import json
import joblib

# Required model package files
required_files = {
    "Model artifact": "../models/aeropure_xgb_v1.joblib",
    "Model metadata": "../models/aeropure_xgb_v1_metadata.json",
    "Feature schema": "../models/aeropure_xgb_v1_feature_schema.json",
    "Drift configuration": "../models/aeropure_xgb_v1_drift_config.json",
    "Drift baseline": "../results/aeropure_xgb_v1_drift_baseline.csv",
    "PSI results": "../results/aeropure_xgb_v1_psi_test.csv",
    "Drift report": "../results/aeropure_xgb_v1_drift_report.csv"
}

print("AeroPure W11 Model Package Verification")
print("=" * 55)

all_files_exist = True

for name, path in required_files.items():
    exists = os.path.exists(path)

    status = "✓ FOUND" if exists else "✗ MISSING"

    print(f"{status:<10} {name}")

    if not exists:
        all_files_exist = False


# Verify model can be loaded
print("\nModel Loading Test")
print("-" * 55)

try:
    verified_model = joblib.load(
        "../models/aeropure_xgb_v1.joblib"
    )

    print("✓ Model loaded successfully")
    print("Model type:", type(verified_model).__name__)

except Exception as e:
    print("✗ Model loading failed")
    print("Error:", e)
    all_files_exist = False


# Verify feature schema
print("\nFeature Schema Test")
print("-" * 55)

try:
    with open(
        "../models/aeropure_xgb_v1_feature_schema.json",
        "r"
    ) as f:
        verified_schema = json.load(f)

    print("✓ Feature schema loaded")
    print("Number of features:", len(verified_schema["features"]))

except Exception as e:
    print("✗ Feature schema loading failed")
    print("Error:", e)
    all_files_exist = False


# Final status
print("\n" + "=" * 55)

if all_files_exist:
    print("✓ AEROPURE W11 MODEL PACKAGE VERIFIED SUCCESSFULLY")
else:
    print("✗ MODEL PACKAGE VERIFICATION FAILED")

print("=" * 55)

AeroPure W11 Model Package Verification
✓ FOUND    Model artifact
✓ FOUND    Model metadata
✓ FOUND    Feature schema
✓ FOUND    Drift configuration
✓ FOUND    Drift baseline
✓ FOUND    PSI results
✓ FOUND    Drift report

Model Loading Test
-------------------------------------------------------
✓ Model loaded successfully
Model type: XGBClassifier

Feature Schema Test
-------------------------------------------------------
✓ Feature schema loaded
Number of features: 17

✓ AEROPURE W11 MODEL PACKAGE VERIFIED SUCCESSFULLY


In [14]:
# Cell 14 — Final AeroPure W11 Summary

w11_summary = {
    "Project": "AeroPure",
    "Week": "W11",
    "Model": MODEL_NAME,
    "Model_Version": MODEL_VERSION,
    "Model_Type": MODEL_TYPE,
    "Champion_Features": len(W8_FEATURES),
    "Training_Rows": len(X_train_w11),
    "Test_Rows": len(X_test_w11),
    "Prediction_Function": "Implemented",
    "Model_Artifact": "Saved and verified",
    "Feature_Schema": "Saved and verified",
    "Drift_Baseline": "Created",
    "PSI_Monitoring": "Implemented",
    "Drift_Configuration": "Saved",
    "Drift_Report": "Created",
    "Significant_Drift_Features": int(significant_count),
    "Moderate_Drift_Features": int(moderate_count),
    "No_Significant_Drift_Features": int(no_drift_count),
    "Overall_Drift_Status": overall_status
}

w11_summary_df = pd.DataFrame(
    list(w11_summary.items()),
    columns=["Item", "Value"]
)

display(w11_summary_df)

# Save final summary
summary_path = "../results/aeropure_w11_summary.csv"

w11_summary_df.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 60)
print("AEROPURE W11 COMPLETED SUCCESSFULLY")
print("=" * 60)
print("Final summary saved to:", summary_path)

,Item,Value
0,Project,AeroPure
1,Week,W11
2,Model,AeroPure XGBoost Champion
3,Model_Version,aeropure_xgb_v1
4,Model_Type,XGBoost
5,Champion_Features,17
6,Training_Rows,15636
7,Test_Rows,8681
8,Prediction_Function,Implemented
9,Model_Artifact,Saved and verified



AEROPURE W11 COMPLETED SUCCESSFULLY
Final summary saved to: ../results/aeropure_w11_summary.csv
